# Bruise segmentation -- AAAI results-section analysis (A100)

Re-evaluates all 5 core models (SegFormer-B2 Teacher, SegFormer-B0 Direct,
SegFormer-B0 Distilled, YOLO26n-sem Direct, YOLO26n-sem Distilled) fresh on
the full 185-image `fixed_consensus_test` set on GPU, then builds every
table/figure needed for the results section: master comparison table,
accuracy-efficiency Pareto figure, YOLO 3-protocol evaluation ablation,
fairness table across ITA skin-tone groups, KD-alpha sweep curves, and a
qualitative prediction grid + failure-case gallery.

**Reuses the same uploaded package as the GPU benchmark notebook** -- no new
upload needed beyond this one notebook file:
```
MyDrive/bruise_segmentation_gpu/bruise_colab_gpu_full.zip
```
Two small supporting tables (ITA skin-tone labels, Optuna KD-alpha trial
history) aren't part of that zip -- they're embedded directly in this
notebook (a few KB each) rather than requiring a second upload.

Results are copied to `MyDrive/bruise_segmentation_gpu/aaai_analysis_<timestamp>/`
at the end.

**Runtime:** Runtime -> Change runtime type -> A100 GPU, before running cell 1.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bruise_segmentation_gpu'
ZIP_PATH = f'{DRIVE_DIR}/bruise_colab_gpu_full.zip'

import os
assert os.path.exists(ZIP_PATH), (
    f'{ZIP_PATH} not found -- this notebook reuses the same package the GPU '
    f'benchmark notebook uses; upload bruise_colab_gpu_full.zip into '
    f'{DRIVE_DIR}/ first if you have not already.'
)
print('Found package:', ZIP_PATH, f'({os.path.getsize(ZIP_PATH) / 1e6:.0f} MB)')

## 2. Confirm GPU
Fails loudly here if the runtime doesn't actually have a GPU attached.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No CUDA GPU visible -- set Runtime > Change runtime type > A100 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))
DEVICE = torch.device('cuda:0')

## 3. Stage package to local disk
Same reasoning as the GPU benchmark notebook: local disk, not the Drive FUSE
mount, so timings are meaningful and disk I/O doesn't bottleneck evaluation.

In [ ]:
import shutil, time

LOCAL_DIR = '/content/bruise_gpu'

t0 = time.time()
shutil.copy(ZIP_PATH, '/content/bruise_colab_gpu_full.zip')
print(f'Copied zip to local disk in {time.time() - t0:.0f}s')

!rm -rf {LOCAL_DIR}
!mkdir -p {LOCAL_DIR}
t0 = time.time()
!unzip -q /content/bruise_colab_gpu_full.zip -d {LOCAL_DIR}
print(f'Unzipped in {time.time() - t0:.0f}s')

## 4. Install dependencies
torch/CUDA are already present on Colab -- do not reinstall torch.

In [ ]:
%cd {LOCAL_DIR}
!pip install -q transformers ultralytics albumentations opencv-python-headless

## 5. Embedded supporting data (ITA skin-tone labels + KD-alpha trial history)

Not part of `bruise_colab_gpu_full.zip` (they're training/labeling artifacts,
not needed for the GPU benchmark) -- embedded here directly instead of a
second upload. `ITA_CSV` is `stem, skin_tone_category` for all 185 fixed-test
images (from `ita_labels/wl_test_per_image_ita.csv`, trimmed to the 2 columns
the fairness table needs). The two alpha-trial tables are each Optuna's full
per-trial history (`optuna_alpha_search/{segformer_b0,yolo_sem}_trials.csv`),
not just the winning value.

In [ ]:
import io
import pandas as pd

ITA_CSV = """stem,skin_tone_category
TAM006_V5_UA_N,Dark (VI)
TAM006_V17_UA_N,Dark (VI)
TAM0009_V2_UA_N,Intermediate (III-IV)
TAM009_V4_UA_N,Intermediate (III-IV)
TAM028_V13_UA_N,Light (II-III)
TAM030_V10_UA_N,Tan (IV)
TAM030_V11_UA_N,Intermediate (III-IV)
TAM030_V12_UA_N,Brown (V)
TAM030_V13_UA_N,Intermediate (III-IV)
TAM031_V1_UA_N,Tan (IV)
TAM031_V2_UA_N,Intermediate (III-IV)
TAM031_V4_UA_N,Tan (IV)
TAM033_V10_UA_N,Tan (IV)
TAM033_V11_UA_N,Tan (IV)
TAM033_V13_UA_N,Tan (IV)
TAM033_V15_UA_N,Dark (VI)
TAM034_V4_UA_N,Intermediate (III-IV)
TAM034_V5_UA_N,Intermediate (III-IV)
TAM034_V9_LA_N,Light (II-III)
TAM034_V11_UA_N,Intermediate (III-IV)
TAM034_V14_UA_N,Intermediate (III-IV)
TAM034_V15_UA_N,Intermediate (III-IV)
TAM034_V17_UA_N,Intermediate (III-IV)
TAM038_V1_UA_N,Dark (VI)
TAM038_V2_UA_N,Intermediate (III-IV)
TAM038_V3_UA_N,Tan (IV)
TAM038_V5_UA_N,Intermediate (III-IV)
TAM038_V11_UA_N,Dark (VI)
TAM038_V12_UA_N,Brown (V)
TAM047_V12_UA_N,Tan (IV)
TAM047_V15_UA_N,Brown (V)
TAM047_V16_UA_N,Intermediate (III-IV)
TAM047_V19_UA_N,Intermediate (III-IV)
TAM047_V20_UA_N,Tan (IV)
TAM057_V9_UA_N,Intermediate (III-IV)
TAM057_V10_UA_N,Tan (IV)
TAM057_V11_LA_N,Intermediate (III-IV)
TAM057_V11_UA_N,Intermediate (III-IV)
TAM057_V12_LA_N,Light (II-III)
TAM057_V13_LA_N,Light (II-III)
TAM057_V13_UA_N,Intermediate (III-IV)
TAM057_V14_LA_N,Light (II-III)
TAM057_V14_UA_N,Dark (VI)
TAM057_V15_UA_N,Light (II-III)
TAM058_V1_UA_N,Light (II-III)
TAM058_V2_UA_N,Light (II-III)
TAM058_V3_UA_N,Light (II-III)
TAM058_V4_UA_N,Light (II-III)
TAM058_V5_UA_N,Light (II-III)
TAM058_V6_UA_N,Light (II-III)
TAM058_V8_UA_N,Light (II-III)
TAM058_V9_UA_N,Light (II-III)
TAM058_V10_UA_N,Light (II-III)
TAM058_V12_UA_N,Light (II-III)
TAM058_V13_UA_N,Intermediate (III-IV)
TAM058_V14_UA_N,Light (II-III)
TAM058_V15_UA_N,Light (II-III)
TAM058_V19_UA_N,Light (II-III)
TAM063_V1_UA_N,Brown (V)
TAM063_V2_UA_N,Brown (V)
TAM063_V3_UA_N,Tan (IV)
TAM063_V4_UA_N,Dark (VI)
TAM063_V5_UA_N,Brown (V)
TAM063_V6_UA_N,Brown (V)
TAM063_V7_UA_N,Brown (V)
TAM063_V8_UA_N,Intermediate (III-IV)
TAM063_V10_UA_N,Dark (VI)
TAM063_V12_UA_N,Tan (IV)
TAM063_V13_UA_N,Dark (VI)
TAM063_V15_UA_N,Brown (V)
TAM063_V16_UA_N,Brown (V)
TAM063_V17_UA_N,Tan (IV)
TAM063_V18_UA_N,Brown (V)
TAM063_V19_UA_N,Dark (VI)
TAM063_V20_UA_N,Dark (VI)
TAM063_V21_UA_N,Dark (VI)
TAM066_V20_UA_N,Light (II-III)
TAM067_V1_UA_N,Dark (VI)
TAM067_V2_UA_N,Dark (VI)
TAM067_V3_UA_N,Dark (VI)
TAM067_V4_UA_N,Brown (V)
TAM067_V5_UA_N,Tan (IV)
TAM067_V6_UA_N,Intermediate (III-IV)
TAM067_V7_UA_N,Brown (V)
TAM067_V8_UA_N,Dark (VI)
TAM067_V9_UA_N,Dark (VI)
TAM067_V10_UA_N,Dark (VI)
TAM067_V11_UA_N,Dark (VI)
TAM069_V9_UA_N,Dark (VI)
TAM076_V3_UA_N,Dark (VI)
TAM076_V4_UA_N,Dark (VI)
TAM076_V5_UA_N,Dark (VI)
TAM076_V6_UA_N,Brown (V)
TAM076_V7_UA_N,Dark (VI)
TAM076_V8_UA_N,Brown (V)
TAM076_V9_UA_N,Dark (VI)
TAM076_V10_UA_N,Dark (VI)
TAM076_V11_UA_N,Dark (VI)
TAM076_V12_UA_N,Dark (VI)
TAM077_V17_UA_N,Intermediate (III-IV)
TAM077_V18_UA_N,Tan (IV)
TAM077_V19_UA_N,Brown (V)
TAM077_V20_UA_N,Dark (VI)
TAM077_V21_UA_N,Intermediate (III-IV)
TAM079_V4_UA_N,Light (II-III)
TAM079_V5_UA_N,Tan (IV)
TAM079_V6_UA_N,Light (II-III)
TAM079_V7_UA_N,Intermediate (III-IV)
TAM079_V8_UA_N,Light (II-III)
TAM079_V9_UA_N,Intermediate (III-IV)
TAM079_V10_UA_N,Intermediate (III-IV)
TAM079_V11_UA_N,Intermediate (III-IV)
TAM079_V12_UA_N,Light (II-III)
TAM080_V2_UA_N,Dark (VI)
TAM080_V3_UA_N,Dark (VI)
TAM080_V4_UA_N,Tan (IV)
TAM080_V5_UA_N,Brown (V)
TAM080_V6_UA_N,Dark (VI)
TAM080_V7_UA_N,Brown (V)
TAM080_V8_UA_N,Brown (V)
TAM080_V10_UA_N,Tan (IV)
TAM080_V14_UA_N,Brown (V)
TAM080_V17_UA_N,Brown (V)
TAM081_V10_UA_N,Dark (VI)
TAM081_V11_UA_N,Dark (VI)
TAM081_V12_UA_N,Dark (VI)
TAM082_V1_UA_N,Brown (V)
TAM082_V2_UA_N,Dark (VI)
TAM082_V3_UA_N,Brown (V)
TAM082_V5_UA_N,Brown (V)
TAM082_V6_UA_N,Brown (V)
TAM083_V1_UA_N,Light (II-III)
TAM083_V2_UA_N,Intermediate (III-IV)
TAM083_V3_UA_N,Brown (V)
TAM083_V4_UA_N,Intermediate (III-IV)
TAM083_V5_UA_N,Tan (IV)
TAM083_V8_UA_N,Brown (V)
TAM083_V9_UA_N,Tan (IV)
TAM084_V2_UA_N,Light (II-III)
TAM084_V3_UA_N,Light (II-III)
TAM084_V5_UA_N,Light (II-III)
TAM084_V6_UA_N,Intermediate (III-IV)
TAM084_V7_UA_N,Tan (IV)
TAM084_V9_UA_N,Light (II-III)
TAM084_V10_UA_N,Intermediate (III-IV)
TAM085_V1_UA_N,Light (II-III)
TAM085_V3_UA_N,Light (II-III)
TAM085_V4_UA_N,Light (II-III)
TAM085_V12_UA_N,Intermediate (III-IV)
TAM085_V13_LA_N,Intermediate (III-IV)
TAM085_V13_UA_N,Intermediate (III-IV)
TAM085_V15_LA_N,Tan (IV)
TAM085_V15_UA_N,Light (II-III)
TAM085_V16_LA_N,Intermediate (III-IV)
TAM085_V16_UA_N,Light (II-III)
TAM085_V17_LA_N,Intermediate (III-IV)
TAM085_V17_UA_N,Light (II-III)
TAM085_V18_UA_N,Light (II-III)
TAM085_V19_LA_N,Light (II-III)
TAM085_V19_UA_N,Light (II-III)
TAM085_V21_LA_N,Light (II-III)
TAM086_V3_UA_N,Intermediate (III-IV)
TAM086_V4_UA_N,Dark (VI)
TAM086_V5_UA_N,Tan (IV)
TAM086_V6_UA_N,Brown (V)
TAM086_V7_UA_N,Dark (VI)
TAM086_V8_UA_N,Tan (IV)
TAM086_V9_UA_N,Dark (VI)
TAM086_V10_UA_N,Dark (VI)
TAM088_V1_UA_N,Dark (VI)
TAM088_V2_UA_N,Dark (VI)
TAM088_V3_UA_N,Dark (VI)
TAM088_V5_UA_N,Dark (VI)
TAM088_V11_UA_N,Dark (VI)
TAM088_V12_UA_N,Dark (VI)
TAM088_V13_UA_N,Dark (VI)
TAM088_V15_UA_N,Dark (VI)
TAM088_V16_UA_N,Dark (VI)
TAM088_V18_UA_N,Dark (VI)
TAM088_V19_UA_N,Dark (VI)
TAM088_V20_UA_N,Dark (VI)
TAM100_V10_UA_N,Dark (VI)
TAM100_V15_UA_N,Dark (VI)
TAM100_V16_UA_N,Dark (VI)
TAM100_V17_UA_N,Brown (V)
"""
ita_df = pd.read_csv(io.StringIO(ITA_CSV))
assert len(ita_df) == 185, f'expected 185 ITA rows, got {len(ita_df)}'

# Optuna KD-alpha trial history -- full per-trial (alpha, val_dice), not just
# the winning value (optuna_alpha_search/{segformer_b0,yolo_sem}_trials.csv).
alpha_trials = {
    'SegFormer-B0': pd.DataFrame({
        'alpha': [0.4, 0.9, 0.7, 0.6, 0.2, 0.2, 0.1, 0.8, 0.6, 0.7, 0.5, 0.5, 0.4, 0.35, 0.35],
        'val_dice': [0.7295957674634986, 0.7112089996678049, 0.7256676901838558, 0.7374641121088935,
                     0.7249021671735879, 0.7276203398483695, 0.7120484197606951, 0.7158298204427621,
                     0.7278752799022585, 0.7221919481155183, 0.7324335932841873, 0.7330592066290751,
                     0.7361785264734217, 0.723933822420343, 0.730360511123314],
    }),
    'YOLO26n-sem': pd.DataFrame({
        'alpha': [0.4, 0.9, 0.7, 0.6, 0.2, 0.2, 0.1, 0.8, 0.6, 0.7, 0.45, 0.35, 0.3, 0.4, 0.3],
        'val_dice': [0.6172451368320132, 0.5992961273817916, 0.5992961273817916, 0.5992961273817916,
                     0.6138990143458514, 0.6138990143458514, 0.6128611858636374, 0.5992961273817916,
                     0.5992961273817916, 0.5992961273817916, 0.6081660920537739, 0.6105806669866881,
                     0.6153561725006205, 0.6074355060201398, 0.6153561725006205],
    }),
}
print('ITA labels:', len(ita_df), 'rows | alpha trials:',
      {k: len(v) for k, v in alpha_trials.items()})

## 6. Load all 5 models

Reuses the same shared pipeline code the local benchmarks/evaluations use
(`pipeline.models.load_segformer_model`, `pipeline.yolo_threshold_temp.load_yolo_model`)
-- same checkpoint + threshold_search.csv resolution as everywhere else in
this project, so numbers here can't drift from how the models are loaded
elsewhere.

In [ ]:
import sys
sys.path.insert(0, LOCAL_DIR)

from pipeline.io_utils import load_yaml
from pipeline.models import load_segformer_model, count_params
from pipeline.yolo_threshold_temp import load_yolo_model
from pipeline.data import load_fixed_test

paths = load_yaml('configs/paths.yaml')
cfg = load_yaml('configs/common_train.yaml')
IMG_H, IMG_W = cfg['img_h'], cfg['img_w']

MODEL_SPECS = [
    {'run_name': 'segformer_b2_teacher', 'display': 'SegFormer-B2 Teacher',
     'family': 'SegFormer', 'kd_family': 'Teacher (no KD)',
     'pretrained_key': 'segformer_b2_pretrained'},
    {'run_name': 'segformer_b0_direct', 'display': 'SegFormer-B0 Direct',
     'family': 'SegFormer', 'kd_family': 'Direct (no KD)',
     'pretrained_key': 'segformer_b0_pretrained'},
    {'run_name': 'segformer_b0_distilled', 'display': 'SegFormer-B0 Distilled',
     'family': 'SegFormer', 'kd_family': 'WL->WL Distilled',
     'pretrained_key': 'segformer_b0_pretrained'},
    {'run_name': 'yolo_sem_direct', 'display': 'YOLO26n-sem Direct',
     'family': 'YOLO', 'kd_family': 'Direct (no KD)'},
    {'run_name': 'yolo_sem_distilled', 'display': 'YOLO26n-sem Distilled',
     'family': 'YOLO', 'kd_family': 'WL->WL Distilled'},
]

MODELS = {}   # run_name -> dict(model/nn, threshold, temperature, params_millions)
for spec in MODEL_SPECS:
    if spec['family'] == 'SegFormer':
        model, threshold, ckpt = load_segformer_model(
            spec['run_name'], spec['pretrained_key'], paths, DEVICE)
        model = model.to(DEVICE).eval()
        n_params, _ = count_params(model)
        MODELS[spec['run_name']] = {'model': model, 'threshold': threshold, 'temperature': 1.0,
                                     'params_millions': n_params / 1e6}
    else:
        nn_model, threshold, temperature, ckpt = load_yolo_model(spec['run_name'], paths, DEVICE)
        n_params = sum(p.numel() for p in nn_model.parameters())
        MODELS[spec['run_name']] = {'model': nn_model, 'threshold': threshold,
                                     'temperature': temperature, 'params_millions': n_params / 1e6}
    print(f"{spec['display']:26s} loaded | threshold={MODELS[spec['run_name']]['threshold']:.3f} "
          f"temperature={MODELS[spec['run_name']]['temperature']:.1f} "
          f"params={MODELS[spec['run_name']]['params_millions']:.2f}M")

test_df = load_fixed_test(paths['fixed_test_manifest'])
print(f'\nFixed test set: {len(test_df)} images')

## 7. Fresh per-image test-set evaluation (all 5 models, GPU)

Dice/IoU/precision/recall/complete-miss per image, computed fresh here (not
reused from any old CSV) -- this is the "evaluate once again" pass. Also
times raw-forward FPS per model on this same GPU for the Pareto figure.

In [ ]:
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader

from pipeline.data import BruiseDataset, get_augmentation
from pipeline.metrics_extended import compute_image_row_extended, summarize_extended
from pipeline.yolo_threshold_temp import yolo_raw_class_logits, bruise_prob_from_logits
from pipeline.track_eval import gpu_timed, build_single_image_tensor

N_BOOTSTRAP = 2000
N_WARMUP, N_ITERS = 30, 100
benchmark_img = test_df.iloc[0]['image_path']

all_per_image = []      # list of DataFrames, one per model, concatenated at the end
all_summaries = []       # one dict per model -> master table

loader = DataLoader(BruiseDataset(test_df, IMG_H, IMG_W, training=False),
                     batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

for spec in MODEL_SPECS:
    run_name, display, family = spec['run_name'], spec['display'], spec['family']
    entry = MODELS[run_name]
    model, thr, temp = entry['model'], entry['threshold'], entry['temperature']
    rows = []
    with torch.no_grad():
        for x, y, stems, img_paths, mask_paths in loader:
            x = x.to(DEVICE, non_blocking=True)
            if family == 'SegFormer':
                logits = model(x)
                prob_np = torch.sigmoid(logits).float().cpu().numpy()
            else:
                class_logits = yolo_raw_class_logits(model, x, out_hw=x.shape[-2:])
                prob_np = bruise_prob_from_logits(class_logits, temp).float().cpu().numpy()
                prob_np = prob_np[:, None, ...]   # match SegFormer's [B,1,H,W] shape below
            gt_np = y.cpu().numpy()
            for i, stem in enumerate(stems):
                pred = (prob_np[i, 0] >= thr).astype('uint8')
                g = (gt_np[i, 0] > 0.5).astype('uint8')
                row = compute_image_row_extended(pred, g, str(stem), surf_dice_delta=2.0)
                row['model'] = display
                rows.append(row)

    per_image_df = pd.DataFrame(rows)
    per_image_df['model'] = display
    all_per_image.append(per_image_df)
    summary = summarize_extended(rows, n_bootstrap=N_BOOTSTRAP)

    # Raw-forward FPS on this GPU, same protocol as the benchmark notebook.
    x_bench = build_single_image_tensor(benchmark_img, cfg, DEVICE)
    if family == 'SegFormer':
        with torch.no_grad():
            fps_mean_ms, _, fps = gpu_timed(lambda: model(x_bench), N_WARMUP, N_ITERS, DEVICE)
    else:
        with torch.no_grad():
            fps_mean_ms, _, fps = gpu_timed(
                lambda: yolo_raw_class_logits(model, x_bench, out_hw=x_bench.shape[-2:]),
                N_WARMUP, N_ITERS, DEVICE)

    all_summaries.append({
        'model': display, 'family': family, 'kd_family': spec['kd_family'],
        'params_millions': entry['params_millions'],
        'threshold_used': thr, 'temperature_used': temp,
        'gpu_raw_fwd_fps': fps, 'gpu_raw_fwd_mean_ms': fps_mean_ms,
        **summary,
    })
    print(f"{display:26s} mean_dice={summary['mean_dice']:.4f} "
          f"median_dice={summary['median_dice']:.4f} "
          f"complete_miss_rate={summary['complete_miss_rate']:.4f} "
          f"| {fps:.1f} FPS")

per_image_all = pd.concat(all_per_image, ignore_index=True)
master = pd.DataFrame(all_summaries)
print(f'\nDone: {len(per_image_all)} per-image rows across {len(MODEL_SPECS)} models.')

## 8. Master comparison table + accuracy-efficiency Pareto figure

In [ ]:
import os
OUT_DIR = f'{LOCAL_DIR}/aaai_analysis'
os.makedirs(OUT_DIR, exist_ok=True)

master_cols = ['model', 'family', 'kd_family', 'params_millions', 'median_dice', 'mean_dice',
               'complete_miss_rate', 'mean_precision', 'mean_recall', 'median_hd95_px',
               'threshold_used', 'temperature_used', 'gpu_raw_fwd_fps']
master_table = master[master_cols].copy()
master_table.to_csv(f'{OUT_DIR}/master_comparison_table.csv', index=False)
per_image_all.to_csv(f'{OUT_DIR}/per_image_all_models.csv', index=False)
master_table

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(7, 5))
markers = {'Teacher (no KD)': '^', 'Direct (no KD)': 'o', 'WL->WL Distilled': 's'}
colors = {'SegFormer': '#1f77b4', 'YOLO': '#d62728'}
for _, row in master_table.iterrows():
    ax.scatter(row['gpu_raw_fwd_fps'], row['median_dice'],
               marker=markers[row['kd_family']], color=colors[row['family']],
               s=120, edgecolor='black', linewidth=0.5, zorder=3)
    ax.annotate(row['model'], (row['gpu_raw_fwd_fps'], row['median_dice']),
                textcoords='offset points', xytext=(6, 6), fontsize=8)
ax.set_xlabel('A100 GPU throughput, raw-forward (FPS)')
ax.set_ylabel('Median Dice (185-image fixed test set)')
ax.set_title('Accuracy vs. GPU throughput -- 5 core models')
ax.grid(True, linestyle='--', alpha=0.4)
family_handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c,
                          markeredgecolor='black', markersize=10, label=fam)
                  for fam, c in colors.items()]
kd_handles = [Line2D([0], [0], marker=m, color='gray', linestyle='', markersize=10, label=k)
              for k, m in markers.items()]
ax.legend(handles=family_handles + kd_handles, loc='lower right', fontsize=8, framealpha=0.9)
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/pareto_fps_vs_dice.png', dpi=200)
plt.show()

## 9. YOLO 3-protocol evaluation ablation

Same checkpoints, 3 postprocessing/calibration protocols, computed fresh (not
reused from any old file):
1. **Native Ultralytics `.predict()` argmax** -- its own internal
   sigmoid+argmax, no threshold/temperature control, at native image
   resolution.
2. **Threshold-only baseline** (temperature=1.0) -- best threshold row at
   T=1.0 from this model's own `threshold_search.csv` (already bundled), applied
   at 640x640.
3. **Threshold + temperature-scaled** (fully calibrated) -- the global best
   (threshold, temperature) row from the same file -- this is the operating
   point used everywhere else in this notebook.

In [ ]:
from ultralytics import YOLO as UltralyticsYOLO
from pipeline.mask_utils import read_gt_mask, yolo_sem_pred_mask
from pipeline.metrics import compute_image_row as compute_image_row_basic
from pipeline.metrics import summarize as summarize_basic

ablation_rows = []
for run_name in ('yolo_sem_direct', 'yolo_sem_distilled'):
    display = next(s['display'] for s in MODEL_SPECS if s['run_name'] == run_name)
    grid = pd.read_csv(f'{run_name}/threshold_search.csv')

    # Protocol 2: threshold-only baseline (T=1.0), best row from the bundled grid.
    t1_rows = grid[grid['temperature'] == 1.0].sort_values('mean_dice', ascending=False)
    t1_thr = float(t1_rows.iloc[0]['threshold'])

    # Protocol 3: fully calibrated (global best) -- same as MODELS[run_name].
    best_thr, best_temp = MODELS[run_name]['threshold'], MODELS[run_name]['temperature']

    nn_model = MODELS[run_name]['model']
    rows_t1, rows_calibrated = [], []
    with torch.no_grad():
        for x, y, stems, img_paths, mask_paths in loader:
            x = x.to(DEVICE, non_blocking=True)
            class_logits = yolo_raw_class_logits(nn_model, x, out_hw=x.shape[-2:])
            prob_t1 = bruise_prob_from_logits(class_logits, 1.0).cpu().numpy()
            prob_cal = bruise_prob_from_logits(class_logits, best_temp).cpu().numpy()
            gt_np = y.cpu().numpy()
            for i, stem in enumerate(stems):
                g = (gt_np[i, 0] > 0.5).astype('uint8')
                rows_t1.append(compute_image_row_basic((prob_t1[i] >= t1_thr).astype('uint8'), g, str(stem)))
                rows_calibrated.append(compute_image_row_basic((prob_cal[i] >= best_thr).astype('uint8'), g, str(stem)))
    summary_t1 = summarize_basic(rows_t1)
    summary_cal = summarize_basic(rows_calibrated)

    # Protocol 1: native Ultralytics .predict(), native image resolution.
    best_pt = f'{run_name}/ultralytics_runs/train/weights/best.pt'
    wrapper = UltralyticsYOLO(best_pt)
    rows_native = []
    for _, r in test_df.iterrows():
        gt = read_gt_mask(r.mask_path).astype('uint8')
        res = wrapper.predict(str(r.image_path), imgsz=IMG_H, device='0', verbose=False)[0]
        pred = yolo_sem_pred_mask(res, gt.shape)
        rows_native.append(compute_image_row_basic(pred, gt, str(r.stem)))
    summary_native = summarize_basic(rows_native)

    for label, summ in [
        ('Native Ultralytics .predict() argmax', summary_native),
        ('Threshold-only (T=1.0)', summary_t1),
        ('Threshold + temperature-scaled (calibrated)', summary_cal),
    ]:
        ablation_rows.append({
            'model': display, 'protocol': label,
            'mean_dice': summ['mean_dice'], 'median_dice': summ.get('median_dice', float('nan')),
            'complete_miss_rate': summ['complete_miss_rate'],
        })
    print(f'{display}: native={summary_native["mean_dice"]:.4f}  '
          f'T=1.0={summary_t1["mean_dice"]:.4f}  calibrated={summary_cal["mean_dice"]:.4f}')

ablation_table = pd.DataFrame(ablation_rows)
ablation_table.to_csv(f'{OUT_DIR}/yolo_protocol_ablation_table.csv', index=False)
ablation_table

## 10. Fairness table (ITA skin-tone groups; DEO/DP analogs)

Joins the fresh per-image results (section 7) against the embedded ITA
labels on `stem`. `deo_recall_range` = max-min mean recall (TPR) across
skin-tone groups, analogous to Difference in Equality of Opportunity;
`dp_miss_rate_range` = max-min complete-miss rate across groups, analogous
to a Demographic-Parity gap for detection.

In [ ]:
merged = per_image_all.merge(ita_df, on='stem', how='inner')
missing = len(per_image_all) - len(merged)
if missing:
    print(f'WARNING: {missing} rows failed to join against the embedded ITA table -- '
          'check for a stem mismatch before trusting the fairness numbers.')

per_group = merged.groupby(['model', 'skin_tone_category']).agg(
    n_images=('stem', 'count'), mean_dice=('dice', 'mean'),
    mean_recall=('recall', 'mean'), complete_miss_rate=('complete_miss', 'mean'),
).reset_index()
per_group.to_csv(f'{OUT_DIR}/fairness_per_group_table.csv', index=False)

summary_rows = []
for model, grp in per_group.groupby('model'):
    summary_rows.append({
        'model': model,
        'deo_recall_range': grp['mean_recall'].max() - grp['mean_recall'].min(),
        'worst_recall_group': grp.loc[grp['mean_recall'].idxmin(), 'skin_tone_category'],
        'best_recall_group': grp.loc[grp['mean_recall'].idxmax(), 'skin_tone_category'],
        'dp_miss_rate_range': grp['complete_miss_rate'].max() - grp['complete_miss_rate'].min(),
        'worst_miss_group': grp.loc[grp['complete_miss_rate'].idxmax(), 'skin_tone_category'],
    })
fairness_summary = pd.DataFrame(summary_rows)
fairness_summary.to_csv(f'{OUT_DIR}/fairness_summary_table.csv', index=False)
fairness_summary

## 11. KD-alpha sweep curves

In [ ]:
for name, df in alpha_trials.items():
    df = df.sort_values('alpha')
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(df['alpha'], df['val_dice'], marker='o', linewidth=1.5)
    best = df.loc[df['val_dice'].idxmax()]
    ax.scatter([best['alpha']], [best['val_dice']], color='red', zorder=5,
               label=f"best: alpha={best['alpha']:.2f}, dice={best['val_dice']:.4f}")
    ax.set_xlabel('KD alpha (distillation loss weight)')
    ax.set_ylabel('Validation mean Dice')
    ax.set_title(f'{name}: KD alpha vs. validation Dice')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(fontsize=8)
    fig.tight_layout()
    safe_name = name.lower().replace('-', '_').replace(' ', '_')
    fig.savefig(f'{OUT_DIR}/alpha_sweep_{safe_name}.png', dpi=200)
    plt.show()

## 12. Qualitative prediction grid + failure-case gallery

A second, small, targeted inference pass (a handful of images x 5 models --
trivial GPU cost) to get actual mask arrays for visualization, rather than
holding all 185x5 masks in memory from section 7.

In [ ]:
import cv2

def predict_mask(run_name, image_rgb):
    entry = MODELS[run_name]
    model, thr, temp = entry['model'], entry['threshold'], entry['temperature']
    tfm = get_augmentation(training=False, img_h=IMG_H, img_w=IMG_W)
    tensor = tfm(image=image_rgb)['image'].float().unsqueeze(0).to(DEVICE)
    family = next(s['family'] for s in MODEL_SPECS if s['run_name'] == run_name)
    with torch.no_grad():
        if family == 'SegFormer':
            prob = torch.sigmoid(model(tensor))[:, 0]
        else:
            class_logits = yolo_raw_class_logits(model, tensor, out_hw=tensor.shape[-2:])
            prob = bruise_prob_from_logits(class_logits, temp)
    return (prob[0].cpu().numpy() >= thr).astype('uint8')


def overlay(image_rgb_640, mask, color=(255, 0, 0), alpha=0.45):
    out = image_rgb_640.copy()
    layer = np.zeros_like(out)
    layer[mask.astype(bool)] = color
    return cv2.addWeighted(layer, alpha, out, 1 - alpha, 0)


def load_and_resize(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (IMG_W, IMG_H))

# --- Qualitative grid: 1 image per ITA skin-tone group, all 5 models ---
groups = ita_df['skin_tone_category'].unique().tolist()
chosen = []
for g in groups:
    stem = ita_df[ita_df['skin_tone_category'] == g]['stem'].iloc[0]
    row = test_df[test_df['stem'] == stem]
    if len(row):
        chosen.append((g, row.iloc[0]))

fig, axes = plt.subplots(len(chosen), len(MODEL_SPECS) + 1,
                          figsize=(3 * (len(MODEL_SPECS) + 1), 3 * len(chosen)))
for row_i, (group, r) in enumerate(chosen):
    img640 = load_and_resize(r['image_path'])
    gt = cv2.resize(read_gt_mask(r['mask_path']).astype('uint8'), (IMG_W, IMG_H),
                     interpolation=cv2.INTER_NEAREST)
    axes[row_i, 0].imshow(overlay(img640, gt, color=(0, 255, 0)))
    axes[row_i, 0].set_ylabel(group, fontsize=9)
    axes[row_i, 0].set_xticks([]); axes[row_i, 0].set_yticks([])
    if row_i == 0:
        axes[row_i, 0].set_title('Ground truth', fontsize=9)
    for col_i, spec in enumerate(MODEL_SPECS, start=1):
        pred = predict_mask(spec['run_name'], cv2.cvtColor(cv2.imread(r['image_path']), cv2.COLOR_BGR2RGB))
        axes[row_i, col_i].imshow(overlay(img640, pred))
        axes[row_i, col_i].set_xticks([]); axes[row_i, col_i].set_yticks([])
        if row_i == 0:
            axes[row_i, col_i].set_title(spec['display'], fontsize=8)
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/qualitative_grid.png', dpi=150)
plt.show()

In [ ]:
# --- Failure gallery: YOLO-direct complete misses, contrasted with SegFormer-B0-direct ---
yolo_direct_rows = per_image_all[per_image_all['model'] == 'YOLO26n-sem Direct']
miss_stems = yolo_direct_rows[yolo_direct_rows['complete_miss']]['stem'].tolist()[:4]

if not miss_stems:
    print('No complete misses found for YOLO26n-sem Direct on this run -- skipping failure gallery.')
else:
    fig, axes = plt.subplots(len(miss_stems), 3, figsize=(9, 3 * len(miss_stems)))
    if len(miss_stems) == 1:
        axes = axes[None, :]
    for row_i, stem in enumerate(miss_stems):
        r = test_df[test_df['stem'] == stem].iloc[0]
        img640 = load_and_resize(r['image_path'])
        gt = cv2.resize(read_gt_mask(r['mask_path']).astype('uint8'), (IMG_W, IMG_H),
                         interpolation=cv2.INTER_NEAREST)
        yolo_pred = predict_mask('yolo_sem_direct', cv2.cvtColor(cv2.imread(r['image_path']), cv2.COLOR_BGR2RGB))
        seg_pred = predict_mask('segformer_b0_direct', cv2.cvtColor(cv2.imread(r['image_path']), cv2.COLOR_BGR2RGB))
        axes[row_i, 0].imshow(overlay(img640, gt, color=(0, 255, 0)))
        axes[row_i, 0].set_ylabel(stem, fontsize=8)
        axes[row_i, 1].imshow(overlay(img640, yolo_pred))
        axes[row_i, 2].imshow(overlay(img640, seg_pred))
        for c in range(3):
            axes[row_i, c].set_xticks([]); axes[row_i, c].set_yticks([])
        if row_i == 0:
            axes[row_i, 0].set_title('Ground truth', fontsize=9)
            axes[row_i, 1].set_title('YOLO26n-sem Direct (miss)', fontsize=9)
            axes[row_i, 2].set_title('SegFormer-B0 Direct', fontsize=9)
    fig.tight_layout()
    fig.savefig(f'{OUT_DIR}/failure_gallery.png', dpi=150)
    plt.show()

## 13. Save everything to Drive

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dest = f'{DRIVE_DIR}/aaai_analysis_{stamp}'
shutil.copytree(OUT_DIR, dest)
print('Saved to:', dest)
!ls -la {dest}